In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    matthews_corrcoef
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


# =================================================
# 1. LOAD DATASET
# =================================================

df = pd.read_csv('/kaggle/input/datasets/harshansk/api-violations-16-8-36/Final_Topology_Dataset_FULL_0_to_5__2.csv')

df = df.sort_values("time_window").reset_index(drop=True)

X = df.drop(columns=['um','dm','time_window','violation_next_window']) \
      .replace([np.inf,-np.inf],np.nan).fillna(0)

y = df["violation_next_window"]

print("Dataset shape:", df.shape)
print("Violation rate:", y.mean())


# =================================================
# 2. TEMPORAL SPLIT (NO SHUFFLING)
# =================================================

n = len(X)

split1 = int(n * 0.95)
split2 = int(split1 * 0.80)

X_train,  y_train  = X.iloc[:split2],       y.iloc[:split2]
X_test,   y_test   = X.iloc[split2:split1], y.iloc[split2:split1]
X_unseen, y_unseen = X.iloc[split1:],       y.iloc[split1:]

print("\nTemporal split")
print("Train :", len(X_train))
print("Test  :", len(X_test))
print("Unseen:", len(X_unseen))


# =================================================
# 3. USE ONLY 20% DATA FOR HYPERPARAMETER SEARCH
# =================================================

sub_size = int(len(X_train) * 0.20)

X_train_small = X_train.iloc[:sub_size]
y_train_small = y_train.iloc[:sub_size]

print("\nUsing", sub_size, "rows for hyperparameter search")


# =================================================
# 4. PIPELINE (SCALER + SVM)
# =================================================

pipeline = Pipeline([

    ("scaler", StandardScaler()),

    ("svm", SVC(
        probability=True,
        class_weight="balanced",
        random_state=42
    ))
])


# =================================================
# 5. TIME SERIES CROSS VALIDATION
# =================================================

tscv = TimeSeriesSplit(n_splits=3)


# =================================================
# 6. HYPERPARAMETER SEARCH
# =================================================

param_grid = {

    "svm__C"     : [0.1, 1, 10],
    "svm__kernel": ["linear","rbf"],
    "svm__gamma" : ["scale","auto"]
}


svm_tuner = RandomizedSearchCV(

    pipeline,

    param_distributions = param_grid,

    n_iter = 6,

    scoring = "f1",

    cv = tscv,

    n_jobs = -1,

    verbose = 1,

    random_state = 42
)

print("\nSearching best SVM parameters...")

svm_tuner.fit(X_train_small, y_train_small)

best_params = svm_tuner.best_params_

print("\nBest Parameters:", best_params)
print("Best CV F1:", svm_tuner.best_score_)


# =================================================
# 7. TRAIN FINAL MODEL ON FULL DATA
# =================================================

print("\nTraining final SVM on FULL training data...")

final_pipeline = Pipeline([

    ("scaler", StandardScaler()),

    ("svm", SVC(

        C = best_params["svm__C"],

        kernel = best_params["svm__kernel"],

        gamma = best_params["svm__gamma"],

        probability=True,

        class_weight="balanced",

        random_state=42
    ))
])


final_pipeline.fit(X_train, y_train)


# =================================================
# 8. THRESHOLD OPTIMIZATION
# =================================================

y_prob_test = final_pipeline.predict_proba(X_test)[:,1]

pre, rec, thr = precision_recall_curve(y_test, y_prob_test)

f1_thr = 2 * pre * rec / (pre + rec + 1e-9)

best_thr = thr[np.argmax(f1_thr[:-1])]

print("\nOptimal threshold:", best_thr)


# =================================================
# 9. PREDICTIONS
# =================================================

y_prob_unseen = final_pipeline.predict_proba(X_unseen)[:,1]

y_pred_test   = (y_prob_test   >= best_thr).astype(int)

y_pred_unseen = (y_prob_unseen >= best_thr).astype(int)

cm = confusion_matrix(y_unseen, y_pred_unseen)


# =================================================
# 10. TEST SET EVALUATION
# =================================================

print("\nTEST SET")

print(classification_report(
    y_test,
    y_pred_test,
    target_names=["Normal","Violation"]
))

print("Balanced Accuracy:",
      balanced_accuracy_score(y_test, y_pred_test))


# =================================================
# 11. UNSEEN SET EVALUATION
# =================================================

print("\nUNSEEN SET")

print(classification_report(
    y_unseen,
    y_pred_unseen,
    target_names=["Normal","Violation"]
))

print("Balanced Accuracy:",
      balanced_accuracy_score(y_unseen, y_pred_unseen))

print("F1:",
      f1_score(y_unseen, y_pred_unseen))

print("Caught:",
      cm[1][1], "/", cm[1].sum())

print("Missed:",
      cm[1][0], "/", cm[1].sum())


# =================================================
# 12. FINAL METRICS TABLE
# =================================================

def full_metrics(y_true, y_pred, y_prob, split):

    cm = confusion_matrix(y_true, y_pred)

    tn,fp,fn,tp = cm.ravel()

    return {

        "Split":split,

        "Precision":round(
            precision_score(y_true,y_pred,zero_division=0),4),

        "Recall":round(
            recall_score(y_true,y_pred,zero_division=0),4),

        "F1 Score":round(
            f1_score(y_true,y_pred,zero_division=0),4),

        "Balanced Accuracy":round(
            balanced_accuracy_score(y_true,y_pred),4),

        "Accuracy":round(
            (tp+tn)/(tp+tn+fp+fn),4),

        "AUC-ROC":round(
            roc_auc_score(y_true,y_prob),4),

        "AUC-PR":round(
            average_precision_score(y_true,y_prob),4),

        "TP":tp,
        "FN":fn,
        "FP":fp,
        "TN":tn,

        "MCC":round(
            matthews_corrcoef(y_true,y_pred),4)
    }


rows = [

    full_metrics(
        y_test,
        y_pred_test,
        y_prob_test,
        "Test"
    ),

    full_metrics(
        y_unseen,
        y_pred_unseen,
        y_prob_unseen,
        "Unseen"
    )
]


summary_df = pd.DataFrame(rows)

print("\nFINAL REPORT")

print(summary_df)


summary_df.to_csv(
"/kaggle/working/svm_final_report.csv",
index=False
)

print("\nSaved: /kaggle/working/svm_final_report.csv")

Dataset shape: (178930, 48)
Violation rate: 0.05129380204549265

Temporal split
Train : 135986
Test  : 33997
Unseen: 8947

Using 27197 rows for hyperparameter search

Searching best SVM parameters...
Fitting 3 folds for each of 6 candidates, totalling 18 fits



Best Parameters: {'svm__kernel': 'linear', 'svm__gamma': 'scale', 'svm__C': 0.1}
Best CV F1: 0.5664859680470054

Training final SVM on FULL training data...



Optimal threshold: 0.10632600846456389



TEST SET
              precision    recall  f1-score   support

      Normal       0.99      0.98      0.98     32235
   Violation       0.65      0.77      0.71      1762

    accuracy                           0.97     33997
   macro avg       0.82      0.87      0.84     33997
weighted avg       0.97      0.97      0.97     33997

Balanced Accuracy: 0.8735910040605253

UNSEEN SET
              precision    recall  f1-score   support

      Normal       0.99      0.98      0.98      8447
   Violation       0.68      0.77      0.72       500

    accuracy                           0.97      8947
   macro avg       0.83      0.87      0.85      8947
weighted avg       0.97      0.97      0.97      8947

Balanced Accuracy: 0.8741085592518054
F1: 0.7202993451824135
Caught: 385 / 500
Missed: 115 / 500

FINAL REPORT
    Split  Precision  Recall  F1 Score  Balanced Accuracy  Accuracy  AUC-ROC  \
0    Test     0.6526  0.7696    0.7063             0.8736    0.9668   0.9690   
1  Unseen     0

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

from sklearn.inspection import permutation_importance
from sklearn.model_selection import learning_curve


# =================================================
# SETUP
# =================================================

save_path = "/kaggle/working/SVM_results"
os.makedirs(save_path, exist_ok=True)


# =================================================
# 1. CONFUSION MATRIX — Test Set
# =================================================

cm_test = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm_test,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Normal','Violation'],
    yticklabels=['Normal','Violation']
)

plt.title(f"SVM Confusion Matrix — Test Set (threshold={best_thr:.2f})")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.savefig(f"{save_path}/svm_confusion_matrix_test.png", dpi=150)
plt.show()

print("1. Confusion Matrix (Test) saved.")


# =================================================
# 2. CONFUSION MATRIX — Unseen Set
# =================================================

cm_unseen = confusion_matrix(y_unseen, y_pred_unseen)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm_unseen,
    annot=True,
    fmt='d',
    cmap='Oranges',
    xticklabels=['Normal','Violation'],
    yticklabels=['Normal','Violation']
)

plt.title(f"SVM Confusion Matrix — Unseen Set (threshold={best_thr:.2f})")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.savefig(f"{save_path}/svm_confusion_matrix_unseen.png", dpi=150)
plt.show()

print("2. Confusion Matrix (Unseen) saved.")


# =================================================
# 3. ROC CURVE
# =================================================

fpr_t, tpr_t, _ = roc_curve(y_test, y_prob_test)
fpr_u, tpr_u, _ = roc_curve(y_unseen, y_prob_unseen)

auc_t = auc(fpr_t, tpr_t)
auc_u = auc(fpr_u, tpr_u)

plt.figure(figsize=(6,5))

plt.plot(fpr_t, tpr_t, label=f"Test AUC = {auc_t:.3f}")
plt.plot(fpr_u, tpr_u, linestyle="--", label=f"Unseen AUC = {auc_u:.3f}")

plt.plot([0,1],[0,1], linestyle=":", color="gray", label="Random")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("SVM ROC Curve")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{save_path}/svm_roc_curve.png", dpi=150)

plt.show()

print("3. ROC Curve saved.")


# =================================================
# 4. PRECISION-RECALL CURVE
# =================================================

pre_t, rec_t, _ = precision_recall_curve(y_test, y_prob_test)
pre_u, rec_u, _ = precision_recall_curve(y_unseen, y_prob_unseen)

pr_auc_t = average_precision_score(y_test, y_prob_test)
pr_auc_u = average_precision_score(y_unseen, y_prob_unseen)

plt.figure(figsize=(6,5))

plt.plot(rec_t, pre_t, label=f"Test AUC-PR = {pr_auc_t:.3f}")
plt.plot(rec_u, pre_u, linestyle="--", label=f"Unseen AUC-PR = {pr_auc_u:.3f}")

plt.axhline(
    y_test.mean(),
    linestyle=":",
    color="gray",
    label=f"Baseline = {y_test.mean():.3f}"
)

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title("SVM Precision-Recall Curve")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{save_path}/svm_precision_recall_curve.png", dpi=150)

plt.show()

print("4. Precision-Recall Curve saved.")


# =================================================
# 5. THRESHOLD ANALYSIS
# =================================================

thresholds = np.arange(0.30,0.80,0.02)

p_list=[]
r_list=[]
f_list=[]

for t in thresholds:

    preds = (y_prob_test >= t).astype(int)

    p_list.append(precision_score(y_test,preds,zero_division=0))
    r_list.append(recall_score(y_test,preds,zero_division=0))
    f_list.append(f1_score(y_test,preds,zero_division=0))


plt.figure(figsize=(8,5))

plt.plot(thresholds,p_list,marker='.',label="Precision")
plt.plot(thresholds,r_list,marker='.',label="Recall")
plt.plot(thresholds,f_list,marker='.',linestyle="--",label="F1")

plt.axvline(x=best_thr,color='red',linestyle=':',label=f"Best threshold={best_thr:.2f}")

plt.xlabel("Threshold")
plt.ylabel("Score")

plt.title("Precision / Recall / F1 vs Threshold")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{save_path}/svm_threshold_metrics.png",dpi=150)

plt.show()

print("5. Threshold metrics saved.")


# =================================================
# 6. FEATURE IMPORTANCE (Permutation)
# =================================================

print("\nCalculating permutation importance...")

perm = permutation_importance(
    final_pipeline,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring='f1'
)

fi_df = pd.DataFrame({
    "Feature":X.columns.tolist(),
    "Importance":perm.importances_mean,
    "Std":perm.importances_std
}).sort_values("Importance",ascending=False)


print("\nTop 20 Features:")
print(fi_df.head(20).to_string(index=False))


top20 = fi_df.head(20)

plt.figure(figsize=(10,8))

sns.barplot(
    x="Importance",
    y="Feature",
    data=top20,
    hue="Feature",
    palette="viridis",
    legend=False
)

plt.errorbar(
    x=top20["Importance"],
    y=range(len(top20)),
    xerr=top20["Std"],
    fmt='none',
    color='black',
    capsize=3
)

plt.title("SVM Top 20 Feature Importance")

plt.tight_layout()
plt.savefig(f"{save_path}/svm_feature_importance_top20.png",dpi=150)

plt.show()


plt.figure(figsize=(10,14))

sns.barplot(
    x="Importance",
    y="Feature",
    data=fi_df,
    hue="Feature",
    palette="viridis",
    legend=False
)

plt.title("SVM All Feature Importance")

plt.tight_layout()
plt.savefig(f"{save_path}/svm_feature_importance_all.png",dpi=150)

plt.show()

print("6. Feature importance saved.")


# =================================================
# 7. LEARNING CURVE
# =================================================

print("\nGenerating learning curve...")

train_sizes, train_scores, val_scores = learning_curve(

    final_pipeline,

    X_train,
    y_train,

    cv=3,

    scoring='f1',

    n_jobs=-1,

    train_sizes=np.linspace(0.1,1.0,5)
)

train_mean = np.mean(train_scores,axis=1)
train_std  = np.std(train_scores,axis=1)

val_mean   = np.mean(val_scores,axis=1)
val_std    = np.std(val_scores,axis=1)

plt.figure(figsize=(7,5))

plt.plot(train_sizes,train_mean,label="Training Score",marker='o')
plt.plot(train_sizes,val_mean,label="Validation Score",marker='o')

plt.fill_between(
    train_sizes,
    train_mean-train_std,
    train_mean+train_std,
    alpha=0.1
)

plt.fill_between(
    train_sizes,
    val_mean-val_std,
    val_mean+val_std,
    alpha=0.1
)

plt.xlabel("Training Size")
plt.ylabel("F1 Score")

plt.title("SVM Learning Curve")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{save_path}/svm_learning_curve.png",dpi=150)

plt.show()

print("7. Learning curve saved.")


# =================================================
# 8. PERFORMANCE SUMMARY
# =================================================

summary_df = pd.DataFrame({

    "Dataset":["Test Set","Unseen Set"],

    "Accuracy":[
        round(accuracy_score(y_test,y_pred_test),4),
        round(accuracy_score(y_unseen,y_pred_unseen),4)
    ],

    "Precision":[
        round(precision_score(y_test,y_pred_test),4),
        round(precision_score(y_unseen,y_pred_unseen),4)
    ],

    "Recall":[
        round(recall_score(y_test,y_pred_test),4),
        round(recall_score(y_unseen,y_pred_unseen),4)
    ],

    "F1-score":[
        round(f1_score(y_test,y_pred_test),4),
        round(f1_score(y_unseen,y_pred_unseen),4)
    ]
})

fig, ax = plt.subplots(figsize=(8,2))

ax.axis('off')

table = ax.table(
    cellText=summary_df.values,
    colLabels=summary_df.columns,
    cellLoc='center',
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.5,2)

plt.title("SVM Performance Summary",pad=20)

plt.tight_layout()
plt.savefig(f"{save_path}/svm_performance_summary.png",dpi=150)

plt.show()

print("\n8. Performance summary saved.")

print("\nAll graphs saved to:", save_path)

for f in sorted(os.listdir(save_path)):
    size = os.path.getsize(f"{save_path}/{f}")
    print(f"{f} — {round(size/1024,2)} KB")